# TakeMeter — Fine-Tuning Notebook

Fine-tunes `distilbert-base-uncased` as a four-label classifier for r/ApplyingToCollege comments, then builds a zero-shot Llama-3.3-70B (Groq) baseline for comparison.

**Labels:** `evidence_based_advice` · `anecdotal_experience` · `unsupported_take` · `emotional_reaction`

**Runtime:** T4 GPU required — Runtime → Change runtime type → T4 GPU

**Notebook flow:**
1. Environment check & installs
2. Load, clean, and split labeled data
3. Tokenize
4. Fine-tune DistilBERT
5. Evaluate the fine-tuned model + **save the model for deployment**
6. Build a Groq zero-shot baseline for comparison
7. Compare baseline vs. fine-tuned, save all results


## 0. Environment check
Confirms a GPU runtime is attached — fine-tuning on CPU works but is far slower.

In [1]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"✓  GPU: {name}  ({mem} GB)")
else:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )


✓  GPU: Tesla T4  (15.6 GB)


Install the libraries this notebook needs beyond Colab's defaults, then import everything up front so failures surface immediately rather than mid-run.

In [2]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "datasets", "scikit-learn", "groq"],
    check=True,
)

import io, os, json, time, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless backend — Colab doesn't need an interactive one
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print(f"✓  PyTorch {torch.__version__}")
print("✓  All imports ready")


✓  PyTorch 2.11.0+cu128
✓  All imports ready


## 1. Label map & data upload

Defines the four target classes, uploads the hand-labeled CSV (`text`, `label` columns), validates it, and produces a stratified 70/15/15 train/val/test split so class proportions stay consistent across all three sets.

In [3]:
# Fixed label ↔ id mapping used everywhere below (training, evaluation, inference).
# Keep this identical to whatever mapping the deployed inference service uses.
LABEL2ID = {
    "evidence_based_advice": 0,
    "anecdotal_experience":  1,
    "unsupported_take":      2,
    "emotional_reaction":    3,
}

ID2LABEL    = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS  = len(LABEL2ID)
LABEL_NAMES = list(LABEL2ID.keys())

print("Label map:")
for label, idx in LABEL2ID.items():
    print(f"  {idx}  {label}")
print(f"\nTotal classes: {NUM_LABELS}")


Label map:
  0  evidence_based_advice
  1  anecdotal_experience
  2  unsupported_take
  3  emotional_reaction

Total classes: 4


In [4]:
from google.colab import files

print("▶  Upload your labeled CSV (required columns: text, label)\n")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw   = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"\n✓  Loaded '{filename}'  —  {len(df_raw):,} rows")
df_raw.head()


▶  Upload your labeled CSV (required columns: text, label)



Saving takemeter_data.csv to takemeter_data.csv

✓  Loaded 'takemeter_data.csv'  —  242 rows


,text,label
0,"Honestly speaking, your extracurriculars are v...",unsupported_take
1,Going from a 2.85 to a 4.0 from one year to th...,unsupported_take
2,For entrepreneurship look at Penn maybe. They ...,unsupported_take
3,Some schools don't consider your freshman year...,evidence_based_advice
4,If you are willing to pay full price your odds...,evidence_based_advice


Basic data hygiene: confirm the required columns exist, drop rows with missing text/label, and reject any label value that isn't in `LABEL2ID` (usually a typo caught early rather than silently mis-training on it).

In [5]:
assert {"text", "label"} <= set(df_raw.columns), \
    f"Missing columns. Found: {df_raw.columns.tolist()}"

before = len(df_raw)
df     = df_raw.dropna(subset=["text", "label"]).copy()
if (dropped := before - len(df)):
    print(f"⚠  Dropped {dropped} rows with null text or label.")

unknown = set(df["label"].unique()) - set(LABEL2ID)
assert not unknown, f"Unknown labels in CSV: {unknown}"

df["label_id"] = df["label"].map(LABEL2ID).astype(int)

print(f"✓  {len(df):,} clean examples\n")
print("Class distribution:")
counts = df["label"].value_counts().reindex(LABEL_NAMES, fill_value=0)
for label, n in counts.items():
    print(f"  {n:>4}  {label:<25}  {'█' * (n // 2)}")


✓  242 clean examples

Class distribution:
    70  evidence_based_advice      ███████████████████████████████████
    58  anecdotal_experience       █████████████████████████████
    76  unsupported_take           ██████████████████████████████████████
    38  emotional_reaction         ███████████████████


**Split strategy:** 70/15/15 train/val/test, stratified by label, fixed `random_state=42` so the split is reproducible across reruns. Stratification matters here because `emotional_reaction` has the fewest examples in the dataset — an unstratified split risks leaving too few (or zero) examples of it in val/test.

In [6]:
RANDOM_SEED = 42

df_trainval, df_test = train_test_split(
    df, test_size=0.15,
    stratify=df["label_id"], random_state=RANDOM_SEED,
)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.15 / 0.85,   # 0.15 of the *original* set, not of trainval
    stratify=df_trainval["label_id"], random_state=RANDOM_SEED,
)

for s in [df_train, df_val, df_test]:
    s.reset_index(drop=True, inplace=True)

total = len(df)
print("Split summary:")
print(f"  Train : {len(df_train):>4}  ({len(df_train)/total:.0%})")
print(f"  Val   : {len(df_val):>4}  ({len(df_val)/total:.0%})")
print(f"  Test  : {len(df_test):>4}  ({len(df_test)/total:.0%})")
print(f"  Total : {total}\n")

print(f"{'Label':<25}  {'Train':>6}  {'Val':>6}  {'Test':>6}")
print("-" * 50)
for label in LABEL_NAMES:
    tr = (df_train["label"] == label).sum()
    va = (df_val["label"]   == label).sum()
    te = (df_test["label"]  == label).sum()
    print(f"  {label:<25}  {tr:>6}  {va:>6}  {te:>6}")
    for split_name, n in [("val", va), ("test", te)]:
        if n < 5:
            print(f"  ⚠  '{label}' has only {n} examples in {split_name} — metrics for this class will be noisy.")

# Persist the raw splits so the baseline section can reload the exact same
# test set later, even if the Colab session restarts in between.
SPLIT_DIR = "/content/takemeter_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
df_train.to_csv(f"{SPLIT_DIR}/train.csv", index=False)
df_val.to_csv(f"{SPLIT_DIR}/val.csv",     index=False)
df_test.to_csv(f"{SPLIT_DIR}/test.csv",   index=False)
print(f"\n✓  Splits saved to {SPLIT_DIR}/")


Split summary:
  Train :  168  (69%)
  Val   :   37  (15%)
  Test  :   37  (15%)
  Total : 242

Label                       Train     Val    Test
--------------------------------------------------
  evidence_based_advice          49      10      11
  anecdotal_experience           40       9       9
  unsupported_take               53      12      11
  emotional_reaction             26       6       6

✓  Splits saved to /content/takemeter_splits/


## 2. Tokenize

Uses **dynamic padding** (`DataCollatorWithPadding`) rather than fixed `max_length` padding: each batch is padded only to its own longest sequence, not to a global max, which meaningfully cuts wasted compute on a mostly-short dataset like this one.

In [7]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(examples):
    # truncation=True protects against the rare long outlier comment;
    # padding is deferred to the data collator (dynamic, per-batch).
    return tokenizer(examples["text"], truncation=True, max_length=256)

def df_to_hf_dataset(split_df):
    ds = Dataset.from_dict({
        "text":   split_df["text"].tolist(),
        "labels": split_df["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True, remove_columns=["text"])

print("Tokenizing …")
ds_train = df_to_hf_dataset(df_train)
ds_val   = df_to_hf_dataset(df_val)
ds_test  = df_to_hf_dataset(df_test)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"✓  Train: {ds_train.shape}  Val: {ds_val.shape}  Test: {ds_test.shape}")

# Sanity-check the max_length=256 choice against the actual data:
# if p95 token length is well under 256, nothing meaningful is being truncated.
lengths = [sum(x) for x in ds_train["attention_mask"]]
print(f"\nToken lengths (train) — "
      f"min={min(lengths)}  median={int(np.median(lengths))}  "
      f"p95={int(np.percentile(lengths, 95))}  max={max(lengths)}")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing …


Map:   0%|          | 0/168 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

✓  Train: (168, 4)  Val: (37, 4)  Test: (37, 4)

Token lengths (train) — min=14  median=46  p95=102  max=182


## 3. Fine-tune DistilBERT

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,   # baked into the model config so a loaded model
    label2id=LABEL2ID,   # can return label names, not just class indices
)
print(f"✓  Model loaded: {MODEL_CHECKPOINT}  ({NUM_LABELS} output labels)")


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓  Model loaded: distilbert-base-uncased  (4 output labels)


Macro-F1, not accuracy, decides which checkpoint is "best." With an imbalanced label distribution (`emotional_reaction` has the fewest examples), accuracy can look fine while the rare class is barely learned — macro-F1 weights all four classes equally.

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  float(accuracy_score(labels, preds)),
        "macro_f1":  float(f1_score(labels, preds, average="macro",
                                    labels=list(range(NUM_LABELS)),
                                    zero_division=0)),
    }


**Hyperparameter notes** (tuned for a small, ~200-example dataset on a single T4):

| Setting | Value | Why |
|---|---|---|
| `num_train_epochs` | 3 | More risks overfitting at this dataset size; raise cautiously if train/val loss are still both dropping. |
| `learning_rate` | 2e-5 | Standard starting point for BERT-family fine-tuning. |
| `per_device_train_batch_size` | 16 | Fits a T4 comfortably; drop to 8 on OOM. |
| `warmup_ratio` | 0.1 | Linear warmup over the first 10% of steps — more stable than a fixed step count when the dataset (and thus total steps) is small. |
| `metric_for_best_model` | `macro_f1` | See note above — matches the metric that actually matters for this imbalanced label set. |
| `fp16` | GPU-conditional | Mixed precision on GPU for speed; automatically off on CPU where it isn't supported the same way. |


In [10]:
training_args = TrainingArguments(
    output_dir="./takemeter-model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning … (5–15 min on T4)")
trainer.train()
print("\n✓  Fine-tuning complete — best checkpoint (by macro-F1) has been reloaded into `trainer.model`.")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning … (5–15 min on T4)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.367401,1.323084,0.378378,0.256579
2,1.279712,1.229756,0.540541,0.401552
3,1.173184,1.189202,0.567568,0.419737


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓  Fine-tuning complete — best checkpoint (by macro-F1) has been reloaded into `trainer.model`.


### Save the fine-tuned model



In [11]:
MODEL_SAVE_DIR = "/content/takemeter-finetuned"

trainer.save_model(MODEL_SAVE_DIR)      # saves model weights + config
tokenizer.save_pretrained(MODEL_SAVE_DIR)  # saves tokenizer alongside it

print(f"✓  Model + tokenizer saved to {MODEL_SAVE_DIR}")
print("Files:", os.listdir(MODEL_SAVE_DIR))

# Zip for download — a single artifact is easier to move into a deployment repo.
ZIP_PATH = "/content/takemeter-finetuned.zip"
shutil.make_archive(ZIP_PATH.replace(".zip", ""), "zip", MODEL_SAVE_DIR)
print(f"✓  Zipped: {ZIP_PATH}")

from google.colab import files as colab_files
colab_files.download(ZIP_PATH)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓  Model + tokenizer saved to /content/takemeter-finetuned
Files: ['tokenizer.json', 'config.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors']
✓  Zipped: /content/takemeter-finetuned.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Optional — push to Hugging Face Hub instead of / in addition to the zip download.**
Deploying from the Hub (rather than committing model weights into a git repo) keeps the deployment repo small and lets Render pull the model at container start. Requires an HF access token with write access, set as `HF_TOKEN` in Colab Secrets.

In [12]:
PUSH_TO_HUB = False   # flip to True and set HF_REPO_ID to actually push
HF_REPO_ID  = "your-username/takemeter-distilbert"

if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    trainer.model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"✓  Pushed to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipped — set PUSH_TO_HUB = True to push to the Hub.")


Skipped — set PUSH_TO_HUB = True to push to the Hub.


## 4. Evaluate the fine-tuned model on the held-out test set

In [13]:
print("Running inference on test set …")
ft_output   = trainer.predict(ds_test)
ft_pred_ids = np.argmax(ft_output.predictions, axis=-1)
ft_true_ids = ft_output.label_ids

ft_probs = torch.nn.functional.softmax(
    torch.tensor(ft_output.predictions), dim=-1
).numpy()

ft_accuracy  = accuracy_score(ft_true_ids, ft_pred_ids)
ft_macro_f1  = f1_score(ft_true_ids, ft_pred_ids, average="macro",
                         labels=list(range(NUM_LABELS)), zero_division=0)
ft_eba_prec  = precision_score(ft_true_ids, ft_pred_ids,
                                labels=[LABEL2ID["evidence_based_advice"]],
                                average="micro", zero_division=0)

print("\n" + "=" * 60)
print("FINE-TUNED MODEL — TEST SET METRICS")
print("=" * 60)
print(f"  Accuracy       : {ft_accuracy:.3f}")
print(f"  Macro-F1       : {ft_macro_f1:.3f}   (target ≥ 0.75)")
print(f"  EBA precision  : {ft_eba_prec:.3f}   (target ≥ 0.80)")

print("\nPer-class metrics:")
print(classification_report(
    ft_true_ids, ft_pred_ids,
    target_names=LABEL_NAMES, zero_division=0,
))


Running inference on test set …



FINE-TUNED MODEL — TEST SET METRICS
  Accuracy       : 0.541
  Macro-F1       : 0.399   (target ≥ 0.75)
  EBA precision  : 0.900   (target ≥ 0.80)

Per-class metrics:
                       precision    recall  f1-score   support

evidence_based_advice       0.90      0.82      0.86        11
 anecdotal_experience       1.00      0.11      0.20         9
     unsupported_take       0.38      0.91      0.54        11
   emotional_reaction       0.00      0.00      0.00         6

             accuracy                           0.54        37
            macro avg       0.57      0.46      0.40        37
         weighted avg       0.63      0.54      0.46        37



In [14]:
RESULTS_DIR = "/content/takemeter_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

cm   = confusion_matrix(ft_true_ids, ft_pred_ids)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Fine-Tuned DistilBERT — Confusion Matrix (Test Set)")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrix_finetuned.png", dpi=150)
plt.show()
print(f"✓  Saved: {RESULTS_DIR}/confusion_matrix_finetuned.png")


✓  Saved: /content/takemeter_results/confusion_matrix_finetuned.png


Surfaces the actual misclassified examples (not just aggregate numbers) so error patterns are inspectable, plus a breakdown by "boundary pair" — which label confusions are highest-cost (e.g. mistaking an unsupported opinion for evidence-based advice) versus lower-stakes mix-ups.

In [15]:
wrong_idx = np.where(ft_pred_ids != ft_true_ids)[0]
print(f"\nMisclassified: {len(wrong_idx)} / {len(ft_true_ids)}\n")

for rank, idx in enumerate(wrong_idx[:15], 1):
    text       = df_test.iloc[idx]["text"]
    true_label = ID2LABEL[ft_true_ids[idx]]
    pred_label = ID2LABEL[ft_pred_ids[idx]]
    confidence = ft_probs[idx][ft_pred_ids[idx]]
    print(f"--- #{rank} ---")
    print(f"True      : {true_label}")
    print(f"Predicted : {pred_label}  (confidence {confidence:.2f})")
    print(f"Text      : {text[:220]}{'…' if len(text) > 220 else ''}")
    print()

# HIGH-COST: mixing up evidence-based advice with an unsupported opinion is the
# most consequential error for this project's stated goal — flagging it separately
# from lower-stakes confusions (like anecdote vs. emotional reaction) makes the
# error analysis easier to prioritize.
BOUNDARY_PAIRS = [
    ("evidence_based_advice", "unsupported_take",    "HIGH-COST"),
    ("anecdotal_experience",  "emotional_reaction",   "low-cost"),
    ("anecdotal_experience",  "evidence_based_advice","medium"),
]
print("Error breakdown by boundary pair:")
for a, b, cost in BOUNDARY_PAIRS:
    a_id, b_id = LABEL2ID[a], LABEL2ID[b]
    ab = ((ft_true_ids == a_id) & (ft_pred_ids == b_id)).sum()
    ba = ((ft_true_ids == b_id) & (ft_pred_ids == a_id)).sum()
    print(f"  [{cost:>9}]  {a:<22} ↔ {b:<22} : {ab+ba} errors  ({ab}+{ba})")

print(f"\n{'Label':<25}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'Support':>8}")
print("-" * 60)
for label in LABEL_NAMES:
    label_id = LABEL2ID[label]
    p = precision_score(ft_true_ids, ft_pred_ids, labels=[label_id], average="micro", zero_division=0)
    r = recall_score(ft_true_ids, ft_pred_ids,    labels=[label_id], average="micro", zero_division=0)
    f = f1_score(ft_true_ids, ft_pred_ids,        labels=[label_id], average="micro", zero_division=0)
    support = sum(1 for t in ft_true_ids if t == label_id)
    flag = "  ← below 0.65" if f < 0.65 else ""
    print(f"  {label:<25}  {p:>6.3f}  {r:>6.3f}  {f:>6.3f}  {support:>8}{flag}")



Misclassified: 17 / 37

--- #1 ---
True      : anecdotal_experience
Predicted : unsupported_take  (confidence 0.29)
Text      : I wrote about overcoming a past ableist mindset in my common app essay as an example of seeing nuance in the world, and ended up getting into Brown.

--- #2 ---
True      : emotional_reaction
Predicted : unsupported_take  (confidence 0.29)
Text      : These past few months been extremely hard for me and many others. It feels like I'm fighting a losing battle with college admissions. The competition it creates has made me resentful and jealous towards classmates and fr…

--- #3 ---
True      : evidence_based_advice
Predicted : unsupported_take  (confidence 0.32)
Text      : We reject around 75% of all 1600's. Lots of these kids have perfect GPAs. So the answer is no and it's not even close.

--- #4 ---
True      : anecdotal_experience
Predicted : unsupported_take  (confidence 0.31)
Text      : my son isn't quite as competitive a student so no Ivy applications,

## 5. Zero-shot baseline (Llama-3.3-70B via Groq)

Gives the fine-tuned DistilBERT a point of comparison against a much larger general-purpose model, prompted with the *same* label definitions used during annotation — so the comparison is apples-to-apples rather than the baseline working from a weaker spec.

In [16]:
from google.colab import userdata
from groq import Groq

# Colab Secrets keeps the key out of the notebook text itself.
# Left sidebar → 🔑 Secrets → add GROQ_API_KEY → enable notebook access.
client_groq = Groq(api_key=userdata.get("GROQ_API_KEY"))
GROQ_MODEL  = "llama-3.3-70b-versatile"
print(f"✓  Groq client ready  ({GROQ_MODEL})")


✓  Groq client ready  (llama-3.3-70b-versatile)


In [17]:
SYSTEM_PROMPT = """You are a classifier for comments from r/ApplyingToCollege.
Classify each comment into exactly one of these four labels:

evidence_based_advice
  A comment that recommends a specific action AND backs it with something
  that would still hold up as fact if the opinion framing were removed —
  a published policy, a school's own reported statistic, or a documented
  practice.

anecdotal_experience
  A comment that mainly recounts the writer's own admissions story —
  stats, decision, timeline — without aiming a recommendation at the reader.

unsupported_take
  A comment that states a confident claim, ranking, prediction, or warning
  — including advice-shaped ones — whose backing would not survive having
  the confident framing stripped away.

emotional_reaction
  A comment that is mainly the writer expressing a feeling about their own
  process, with little or no specific detail or reasoning behind it.

Decision rule for evidence_based_advice vs unsupported_take:
Strip the imperative and isolate the justification on its own.
If what remains is a specific, sourced fact (a CDS range, a published
deadline, a documented mechanism), output evidence_based_advice.
If what remains is an unfalsifiable claim about how schools treat
applicants, output unsupported_take — regardless of how directive or
numerically precise the comment sounds.

Output ONLY the label name. No explanation, no punctuation, no extra text.
Valid outputs: evidence_based_advice | anecdotal_experience | unsupported_take | emotional_reaction"""

print(f"System prompt ready ({len(SYSTEM_PROMPT)} chars)")


System prompt ready (1524 chars)


Reloads the test split from disk (rather than reusing the in-memory `df_test`) so the baseline is guaranteed to run on the exact same examples even if this section is re-run in a fresh session. `temperature=0` for deterministic, reproducible labels.

In [18]:
df_test_bl  = pd.read_csv(f"{SPLIT_DIR}/test.csv")
bl_results  = []
bl_unparseable = []

print(f"Classifying {len(df_test_bl)} test examples …\n")

for i, row in df_test_bl.iterrows():
    try:
        resp = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": row["text"]},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = resp.choices[0].message.content.strip().lower()
    except Exception as e:
        raw = ""
        print(f"  Row {i}: API error — {e}")

    # Match the longest label name first so e.g. "unsupported_take" isn't
    # accidentally matched by a shorter label that happens to be a substring.
    predicted = None
    for label in sorted(LABEL_NAMES, key=len, reverse=True):
        if label in raw:
            predicted = label
            break

    if predicted is None:
        bl_unparseable.append({"index": i, "raw": raw, "text": row["text"][:80]})
        predicted = "__unparseable__"

    bl_results.append({
        "text":      row["text"],
        "true":      row["label"],
        "predicted": predicted,
    })

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df_test_bl)} done")
    time.sleep(0.1)  # light throttling, comfortably under Groq's free-tier rate limit

print(f"\n✓  Done  —  unparseable: {len(bl_unparseable)}/{len(df_test_bl)}")
if len(bl_unparseable) / len(df_test_bl) > 0.10:
    print("⚠  >10% unparseable — revise the system prompt and re-run.")


Classifying 37 test examples …

  10/37 done
  20/37 done
  30/37 done

✓  Done  —  unparseable: 0/37


In [19]:
df_bl = pd.DataFrame(bl_results)
df_bl_scored = df_bl[df_bl["predicted"] != "__unparseable__"].copy()

bl_true = df_bl_scored["true"].tolist()
bl_pred = df_bl_scored["predicted"].tolist()

bl_accuracy = accuracy_score(bl_true, bl_pred)
bl_macro_f1 = f1_score(bl_true, bl_pred, average="macro",
                        labels=LABEL_NAMES, zero_division=0)
bl_eba_prec = precision_score(bl_true, bl_pred,
                               labels=["evidence_based_advice"],
                               average="micro", zero_division=0)

print("\n" + "=" * 60)
print("BASELINE METRICS (zero-shot Groq)")
print("=" * 60)
print(f"  Accuracy       : {bl_accuracy:.3f}")
print(f"  Macro-F1       : {bl_macro_f1:.3f}   (target ≥ 0.75)")
print(f"  EBA precision  : {bl_eba_prec:.3f}   (target ≥ 0.80)")
print()
print(classification_report(bl_true, bl_pred, labels=LABEL_NAMES, zero_division=0))

cm_bl   = confusion_matrix(bl_true, bl_pred, labels=LABEL_NAMES)
disp_bl = ConfusionMatrixDisplay(confusion_matrix=cm_bl, display_labels=LABEL_NAMES)
fig2, ax2 = plt.subplots(figsize=(7, 5))
disp_bl.plot(ax=ax2, cmap="Oranges", colorbar=False)
ax2.set_title("Groq Baseline — Confusion Matrix (Test Set)")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrix_baseline.png", dpi=150)
plt.show()
print(f"✓  Saved: {RESULTS_DIR}/confusion_matrix_baseline.png")



BASELINE METRICS (zero-shot Groq)
  Accuracy       : 0.838
  Macro-F1       : 0.826   (target ≥ 0.75)
  EBA precision  : 0.900   (target ≥ 0.80)

                       precision    recall  f1-score   support

evidence_based_advice       0.90      0.82      0.86        11
 anecdotal_experience       0.82      1.00      0.90         9
     unsupported_take       0.82      0.82      0.82        11
   emotional_reaction       0.80      0.67      0.73         6

             accuracy                           0.84        37
            macro avg       0.83      0.83      0.83        37
         weighted avg       0.84      0.84      0.83        37

✓  Saved: /content/takemeter_results/confusion_matrix_baseline.png


## 6. Compare baseline vs. fine-tuned, save everything

A guard cell first: if this section is run in a new session where Section 5's variables no longer exist in memory, it reloads the baseline numbers from the CSV saved earlier instead of failing.

In [20]:
try:
    bl_accuracy
except NameError:
    print("⚠  Baseline variables not found — reloading from saved predictions …")
    _bl = pd.read_csv(f"{RESULTS_DIR}/baseline_predictions.csv")
    _bl_scored = _bl[_bl["predicted"] != "__unparseable__"]
    bl_true = _bl_scored["true"].tolist()
    bl_pred = _bl_scored["predicted"].tolist()
    bl_accuracy = accuracy_score(bl_true, bl_pred)
    bl_macro_f1 = f1_score(bl_true, bl_pred, average="macro",
                            labels=LABEL_NAMES, zero_division=0)
    bl_eba_prec = precision_score(bl_true, bl_pred,
                                  labels=["evidence_based_advice"],
                                  average="micro", zero_division=0)
    bl_unparseable = []
    print("✓  Baseline numbers restored from baseline_predictions.csv")


In [21]:
print("\nPer-class F1 — baseline vs fine-tuned:")
print(f"  {'Label':<25}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Delta':>8}")
print("  " + "-" * 58)

for label in LABEL_NAMES:
    label_id = LABEL2ID[label]
    bl_f  = f1_score(
        [LABEL2ID[t] for t in bl_true],
        [LABEL2ID[p] for p in bl_pred],
        labels=[label_id], average="micro", zero_division=0,
    )
    ft_f  = f1_score(
        ft_true_ids, ft_pred_ids,
        labels=[label_id], average="micro", zero_division=0,
    )
    delta = ft_f - bl_f
    sign  = "+" if delta >= 0 else ""
    flag  = "  ← below 0.65" if ft_f < 0.65 else ""
    print(f"  {label:<25}  {bl_f:>10.3f}  {ft_f:>12.3f}  {sign}{delta:>7.3f}{flag}")



Per-class F1 — baseline vs fine-tuned:
  Label                        Baseline    Fine-tuned     Delta
  ----------------------------------------------------------
  evidence_based_advice           0.857         0.857  +  0.000
  anecdotal_experience            0.900         0.200   -0.700  ← below 0.65
  unsupported_take                0.818         0.541   -0.278  ← below 0.65
  emotional_reaction              0.727         0.000   -0.727  ← below 0.65


Writes every prediction, metric, and plot to disk as a single results bundle, then downloads it — this is the artifact set to hand in or reference when writing up results.

In [22]:
df_bl.to_csv(f"{RESULTS_DIR}/baseline_predictions.csv",   index=False)
df_bl[df_bl["true"] != df_bl["predicted"]].to_csv(
    f"{RESULTS_DIR}/baseline_errors.csv", index=False)

pd.DataFrame({
    "text":           df_test["text"].tolist(),
    "true":           [ID2LABEL[i] for i in ft_true_ids],
    "predicted":      [ID2LABEL[i] for i in ft_pred_ids],
    "confidence":     [ft_probs[i][ft_pred_ids[i]] for i in range(len(ft_pred_ids))],
}).to_csv(f"{RESULTS_DIR}/finetuned_predictions.csv", index=False)

results_json = {
    "model_checkpoint":      MODEL_CHECKPOINT,
    "groq_model":            GROQ_MODEL,
    "test_set_size":         len(df_test),
    "label_map":             LABEL2ID,
    "baseline": {
        "accuracy":      round(bl_accuracy,  4),
        "macro_f1":      round(bl_macro_f1,  4),
        "eba_precision": round(bl_eba_prec,   4),
        "unparseable":   len(bl_unparseable),
    },
    "finetuned": {
        "accuracy":      round(ft_accuracy,  4),
        "macro_f1":      round(ft_macro_f1,  4),
        "eba_precision": round(ft_eba_prec,   4),
    },
    "delta": {
        "accuracy":      round(ft_accuracy - bl_accuracy,  4),
        "macro_f1":      round(ft_macro_f1  - bl_macro_f1, 4),
        "eba_precision": round(ft_eba_prec   - bl_eba_prec, 4),
    },
}
with open(f"{RESULTS_DIR}/evaluation_results.json", "w") as f:
    json.dump(results_json, f, indent=2)

print(f"\n✓  All outputs saved to {RESULTS_DIR}/")
for fname in [
    "baseline_predictions.csv", "baseline_errors.csv",
    "finetuned_predictions.csv",
    "confusion_matrix_baseline.png", "confusion_matrix_finetuned.png",
    "evaluation_results.json",
]:
    print(f"   {fname}")

from google.colab import files as colab_files
for fname in [
    "baseline_predictions.csv", "baseline_errors.csv",
    "finetuned_predictions.csv",
    "confusion_matrix_baseline.png", "confusion_matrix_finetuned.png",
    "evaluation_results.json",
]:
    colab_files.download(f"{RESULTS_DIR}/{fname}")



✓  All outputs saved to /content/takemeter_results/
   baseline_predictions.csv
   baseline_errors.csv
   finetuned_predictions.csv
   confusion_matrix_baseline.png
   confusion_matrix_finetuned.png
   evaluation_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>